In [0]:
from pyspark.sql.functions import col, lpad, trim, date_format, from_utc_timestamp
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("year", "2025", "Year (YYYY)")
dbutils.widgets.text("month", "01", "Month (MM)")
dbutils.widgets.text("debug", "False", "Debug Mode")

year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")
debug = dbutils.widgets.get("debug")

if debug == "False":
  debug = False
else:
  debug = True

source_table = "oftalmo_sus.01_bronze.tb_datasus_sia"
target_table = "oftalmo_sus.02_silver.tb_fato_sia"

print(f"Month: {month}, Year: {year}, Debug Mode: {debug}\nSource Table: {source_table}\nTarget Table: {target_table}")

In [0]:
sia_pa_data = spark.read.table(source_table).filter(f"batch_processing = '{year}{month}'")

if debug:
    display(sia_pa_data.limit(10))

In [0]:
sia_pa_data = sia_pa_data.withColumn(
    "PA_PROC_ID_CLEAN",
    lpad(trim(col("PA_PROC_ID").cast("string")), 10, "0")
)

In [0]:
if debug:
    from pyspark.sql.functions import col

    # 1. Lendo o dado bruto diretamente do Unity Catalog
    df_bronze_sia = sia_pa_data

    print("📊 1. Total geral de atendimentos na Bronze:", df_bronze_sia.count())

    # 2. Buscando oftalmologia usando o LIKE (que ignora a tipagem, buscando em qualquer parte da string)
    df_oftalmo_bronze = df_bronze_sia.filter(
        col("PA_PROC_ID").like("%0405%") | 
        col("PA_PROC_ID").like("%03010%") | 
        col("PA_PROC_ID").like("%021106%")
    )

    print("👁️ 2. Total de oftalmologia encontrada na base CRUA:", df_oftalmo_bronze.count())

    # 3. Verificando se a coluna sofreu corrupção grave
    print("⚠️ 3. Total de procedimentos NULOS ou VAZIOS:", 
        df_bronze_sia.filter(col("PA_PROC_ID").isNull() | (col("PA_PROC_ID") == "")).count())

    # 4. Amostra real do formato que veio do governo
    print("🔍 4. Amostra do PA_PROC_ID original:")
    display(df_bronze_sia.select("PA_PROC_ID").distinct().limit(5))

In [0]:
(
    sia_pa_data.write 
    .format("delta")
    .mode("append")
    .partitionBy("batch_processing")
    .option("mergeSchema", "true") # Permite evolução automática do schema
    .saveAsTable(target_table)
)

print("✅ Camada Silver concluída! Tabela Fato salva com sucesso.")

In [0]:
delta_table = DeltaTable.forName(spark, target_table)

history_df = delta_table.history()

display(
    history_df
    .orderBy(col("version").desc())
    .limit(1)
    .select(
        col("version"),
        date_format(from_utc_timestamp(col("timestamp"), "America/Sao_Paulo"), "dd-MM-yyyy HH:mm:ss").alias("timestamp"),
        col("operation"),
        col("operationMetrics.insert").alias("insert"),
        col("operationMetrics.delete").alias("delete"),
        col("operationMetrics.update").alias("update"),
        col("operationMetrics.numOutputRows").alias("num_rows")
    )
)